# Atari DQN Training
Runs the DQN training loop from the DeepMind 2015 paper on Atari Breakout.

In [2]:
!nvidia-smi

Sun Jun 28 11:46:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import psutil

ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total/1e9:.1f}GB")
print(f"Available: {ram.available/1e9:.1f}GB")

Total RAM: 56.9GB
Available: 55.3GB


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
!git pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 44.91 KiB | 2.50 MiB/s, done.
From https://github.com/pradsgit/atari-dqn
   c8e657f..0d09d08  main       -> origin/main
Updating c8e657f..0d09d08
Fast-forward
 agent.py                       |    2 +-
 evaluate.py => dqn_evaluate.py |    0
 pyproject.toml                 |    1 +
 run_training.ipynb             | 4097 +++++++++++++++++++++++++++++++++++++++-
 train.py                       |    3 +-
 uv.lock                        |    2 +
 6 files changed, 4090 insertions(+), 15 deletions(-)
 rename evaluate.py => dqn_evaluate.py (100%)


In [5]:
import os

if not os.path.exists('/content/atari-dqn'):
    !git clone https://github.com/pradsgit/atari-dqn.git /content/atari-dqn

os.chdir('/content/atari-dqn')
print('Working directory:', os.getcwd())

Cloning into '/content/atari-dqn'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 40 (delta 15), reused 35 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 124.00 KiB | 24.80 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Working directory: /content/atari-dqn


In [7]:
!pip install -q "gymnasium[atari]" ale-py opencv-python torch torchvision

In [8]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch: 2.11.0+cu128
CUDA available: True
Device: cuda


In [14]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import train

train.MAX_STEPS       = 5_000_000
train.MIN_REPLAY_SIZE = 10_000
train.EPSILON_DECAY   = 500_000
train.REPLAY_SIZE     = 100_000
train.CHECKPOINT_DIR  = "/content/drive/MyDrive/atari-dqn/checkpoints"

In [10]:
print(f"REPLAY_SIZE: {train.REPLAY_SIZE}")
print(f"MIN_REPLAY_SIZE: {train.MIN_REPLAY_SIZE}")

REPLAY_SIZE: 100000
MIN_REPLAY_SIZE: 10000


In [ ]:
import tracemalloc
tracemalloc.start()

# run training
train.train()

snapshot = tracemalloc.take_snapshot()
for stat in snapshot.statistics('lineno')[:10]:
    print(stat)

using device: cuda
episode    1 | steps      132 | reward 0.0 | epsilon 1.000 | loss collecting | ram 1.6GB
episode    2 | steps      369 | reward 2.0 | epsilon 0.999 | loss collecting | ram 1.6GB
episode    3 | steps      575 | reward 2.0 | epsilon 0.999 | loss collecting | ram 1.6GB
episode    4 | steps      709 | reward 0.0 | epsilon 0.999 | loss collecting | ram 1.6GB
episode    5 | steps      928 | reward 3.0 | epsilon 0.998 | loss collecting | ram 1.6GB
episode    6 | steps     1096 | reward 1.0 | epsilon 0.998 | loss collecting | ram 1.8GB
episode    7 | steps     1366 | reward 3.0 | epsilon 0.998 | loss collecting | ram 1.8GB
episode    8 | steps     1535 | reward 1.0 | epsilon 0.997 | loss collecting | ram 1.8GB
episode    9 | steps     1814 | reward 3.0 | epsilon 0.997 | loss collecting | ram 1.9GB
episode   10 | steps     1944 | reward 0.0 | epsilon 0.997 | loss collecting | ram 1.9GB
episode   11 | steps     2140 | reward 2.0 | epsilon 0.996 | loss collecting | ram 1.9GB
ep

In [31]:
import importlib.util, traceback
spec = importlib.util.spec_from_file_location("dqn_evaluate", "/content/atari-dqn/dqn_evaluate.py")
mod = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(mod)
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_3987/2594089552.py", line 5, in <cell line: 0>
    spec.loader.exec_module(mod)
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/content/atari-dqn/dqn_evaluate.py", line 5, in <module>
    from env import AtariEnv
ModuleNotFoundError: No module named 'env'


In [29]:
import os, sys

if not os.path.exists('/content/atari-dqn'):
    os.system('git clone https://github.com/pradsgit/atari-dqn.git /content/atari-dqn')

os.chdir('/content/atari-dqn')
if '/content/atari-dqn' not in sys.path:
    sys.path.insert(0, '/content/atari-dqn')

import dqn_evaluate as evaluate
import torch

evaluate.CHECKPOINT_DIR = "/content/drive/MyDrive/atari-dqn/checkpoints"
evaluate.VIDEO_DIR      = "/content/drive/MyDrive/atari-dqn/videos"

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoints = sorted([f for f in os.listdir(evaluate.CHECKPOINT_DIR) if f.endswith(".pt")])
print(f"found {len(checkpoints)} checkpoints:")
for c in checkpoints:
    print(" ", c)

latest = os.path.join(evaluate.CHECKPOINT_DIR, checkpoints[-1])
agent = evaluate.load_agent(latest, device)

mean, std = evaluate.evaluate(agent, n_episodes=10)
evaluate.record_video(agent, n_episodes=1)

ModuleNotFoundError: No module named 'dqn_evaluate'